# Stage 6 — A/B/C Variant Comparison (Fresh Graduate Resume)

**Represents 5 of the 20 required test cases** (this resume × 5 jobs). Second of four resume-based batches (Trixie Mok, Alex Chen, John Doe, +1 more), 5 test cases each, totalling 20.

Same real experiment as the Trixie resume test, run against a different candidate profile: `fresh_graduate_resume.pdf` — Alex Chen, final-year BBA (Marketing & Analytics), no paid work experience, only academic/leadership projects.

- **A — Minimal LLM**: bare prompt, "Rewrite this resume for this job."
- **B — Simplified system**: "Extract relevant skills and tailor this resume to the job" — no schema, no anti-fabrication instructions.
- **C — Full system**: the actual deployed app logic (`JobPortalService._generate_tailored_resume`), structured JSON schema, evidence required per claim, explicit instruction never to invent credentials/employers/dates/metrics/skills.

Same 5 jobs and same 4 criteria as the Trixie test, for direct comparability:

| Criterion | Question |
|---|---|
| **Grounding / No Fabrication** | Does the output rely only on facts present in the source resume? |
| **Personalization / Relevance** | Does it tailor emphasis to the job without force-fitting irrelevant claims? |
| **Correctness & Completeness** | Are the facts accurate and complete? |
| **Clarity & Change Evidence** | Is the output well-structured, explaining what changed and why? |

**This is a retest.** The results below are current — regenerated after the team shipped a deterministic anti-fabrication check to System C (`_enforce_fidelity`). The original pre-fix results are preserved as `*_PREFIX.json` for direct before/after comparison (see the Retest section near the end).

In [1]:
import json
from pathlib import Path

HERE = Path(".")
naive = json.loads((HERE / "variant_comparison_results.json").read_text(encoding="utf-8"))
grounded = json.loads((HERE / "variant_comparison_results_grounded_judge.json").read_text(encoding="utf-8"))

CRITERIA = ["grounding", "personalization", "correctness", "clarity"]
print(f"Loaded {len(naive)} test cases (naive judge) and {len(grounded)} (grounding-gated judge)")

Loaded 5 test cases (naive judge) and 5 (grounding-gated judge)


## Round 1: Naive LLM-judge scoring

In [2]:
def print_table(results, score_key):
    header = f"{'Job':<22}{'Variant':<8}" + "".join(f"{c[:10]:<12}" for c in CRITERIA) + "avg"
    print(header)
    print("-" * len(header))
    totals = {v: {c: [] for c in CRITERIA} for v in "ABC"}
    for entry in results:
        scores_block = entry[score_key]
        for v in "ABC":
            row = scores_block[v]
            vals = [row[c]["score"] for c in CRITERIA]
            avg = sum(vals) / len(vals)
            for c, val in zip(CRITERIA, vals):
                totals[v][c].append(val)
            print(f"{entry['job_title'][:21]:<22}{v:<8}" + "".join(f"{val:<12}" for val in vals) + f"{avg:.2f}")
    print()
    print("OVERALL AVERAGES")
    for v in "ABC":
        per_c = {c: sum(totals[v][c]) / len(totals[v][c]) for c in CRITERIA}
        overall = sum(per_c.values()) / len(per_c)
        print(f"  {v}: " + ", ".join(f"{c}={val:.2f}" for c, val in per_c.items()) + f"  -> overall={overall:.2f}")

print_table(naive, "scores")

Job                   Variant grounding   personaliz  correctnes  clarity     avg
---------------------------------------------------------------------------------
Software Engineer     A       2           4           3           4           3.25
Software Engineer     B       5           5           5           5           5.00
Software Engineer     C       5           3           5           2           3.75
Machine Learning      A       2           4           3           4           3.25
Machine Learning      B       4           4           4           4           4.00
Machine Learning      C       5           3           5           5           4.50
DevOps Engineer       A       1           4           2           3           2.50
DevOps Engineer       B       2           4           3           4           3.25
DevOps Engineer       C       5           2           5           1           3.25
Database Administrato A       4           5           5           5           4.75
Databa

**C still wins narrowly here, as in the pre-fix run** — its perfect grounding (5.0 on every job) offsets a low personalization score. A candidate with zero technical background genuinely doesn't fit roles like DevOps Engineer or Full Stack Developer, and C tells the truth about that instead of inventing a fit.

In [3]:
entry = [e for e in naive if e["job_title"] == "Full Stack Developer"][0]
print("Job:", entry["job_title"], "@", entry["company_name"])
print()
print("--- Variant A output (excerpt) ---")
print(entry["outputs"]["A"][:600])
print()
print("--- Variant C output (summary + skills) ---")
c = entry["outputs"]["C"]
print("summary:", c["resume"].get("summary"))
print("skills:", c["resume"].get("skills"))

Job: Full Stack Developer @ InnoWave Networks

--- Variant A output (excerpt) ---
Here's a rewritten version of the resume tailored to the job description:

**ALEX CHEN**
Singapore | +65 9123 4567 | [redacted-email] | linkedin.com/in/alexchen

**PROFESSIONAL SUMMARY**
Highly motivated and detail-oriented full-stack developer with a strong foundation in computer science and a passion for transforming clinical trial workflows. Proficient in a range of technologies, including Mean Stack (MongoDB, AngularJS, Express, Node.js), Java, Postgres, Redis, RabbitMQ, and Elasticsearch. Excited about the prospect of developing and testing cutting-edge systems that leverage electronic he

--- Variant C output (summary + skills) ---
summary: Motivated and detail-oriented recent graduate with a Bachelor of Business Administration, demonstrated strong leadership, analytical skills, and project management capabilities through academic campaigns and university club leadership.
skills: ['Microsoft Office 

## Round 2: Grounding-gated judge (the fix)

Same hard rule as the Trixie test: any fabricated skill/employer/metric caps that variant's scores, regardless of fluency.

In [4]:
print_table(grounded, "grounded_judge_scores")

Job                   Variant grounding   personaliz  correctnes  clarity     avg
---------------------------------------------------------------------------------
Software Engineer     A       5           4           5           4           4.50
Software Engineer     B       5           5           5           5           5.00
Software Engineer     C       5           4           5           4           4.50
Machine Learning      A       1           2           2           2           1.75
Machine Learning      B       5           4           5           4           4.50
Machine Learning      C       5           3           5           5           4.50
DevOps Engineer       A       1           2           2           2           1.75
DevOps Engineer       B       1           2           2           2           1.75
DevOps Engineer       C       5           4           5           5           4.75
Database Administrato A       5           4           5           4           4.50
Databa

In [5]:
# Fabrications the grounding-gated judge actually caught, per variant
for entry in grounded:
    fab = entry["grounded_judge_scores"].get("fabrications", {})
    if any(fab.get(v) for v in "ABC"):
        print(entry["job_title"], "@", entry["company_name"])
        for v in "ABC":
            items = [i for i in (fab.get(v) or []) if i]
            if items:
                print(f"  {v}: {items}")
        print()

Software Engineer @ WestGate Networks

Machine Learning @ LumaCore Data
  A: ['Python programming language (basic proficiency)', 'statistical concepts', 'machine learning principles']

DevOps Engineer @ CloudHarbor Labs
  A: ['Programming languages: Python, C# (familiarity with C# and willingness to learn)', 'Scripting languages: Shell Scripting, PowerShell (familiarity with PowerShell and willingness to learn)', 'Familiarity with agile development methodologies and version control systems', 'Strong understanding of operating systems: Windows, Linux']
  B: ['Results-driven and analytical professional with experience in project management, data analysis, and communication']

Database Administrator @ Solstice Digital

Full Stack Developer @ InnoWave Networks
  A: ['Mean Stack (MongoDB, AngularJS, Express, Node.js)', 'Java', 'Postgres', 'Redis', 'RabbitMQ', 'Elasticsearch', 'Cloud service providers (e.g., AWS, Google Cloud)', 'Familiarity with SQL queries and NoSQL databases', 'Experience

## Retest: Before vs. After the Anti-Fabrication Fix

Unlike Trixie's or John Doe's resume, C had **zero fabrications on this resume both before and after the fix** — there was nothing here for `_enforce_fidelity` to catch. This is still a useful retest: it confirms the fix didn't regress anything, and confirms the original candidate-dependent-fabrication finding is stable rather than a one-off.

In [6]:
prefix_grounded = json.loads((HERE / "variant_comparison_results_grounded_judge_PREFIX.json").read_text(encoding="utf-8"))

def overall_by_variant(results, score_key):
    out = {}
    for v in "ABC":
        vals = []
        for entry in results:
            row = entry[score_key][v]
            vals.append(sum(row[c]["score"] for c in CRITERIA) / len(CRITERIA))
        out[v] = sum(vals) / len(vals)
    return out

before = overall_by_variant(prefix_grounded, "grounded_judge_scores")
after = overall_by_variant(grounded, "grounded_judge_scores")

print(f"{'Variant':<20}{'Before fix':<14}{'After fix':<14}")
for v, label in zip("ABC", ["A - Minimal", "B - Simplified", "C - Full system"]):
    print(f"{label:<20}{before[v]:<14.2f}{after[v]:<14.2f}")

print()
def fab_jobs_count(results, variant):
    n = 0
    for entry in results:
        items = [i for i in (entry["grounded_judge_scores"].get("fabrications", {}) or {}).get(variant, []) if i]
        if items:
            n += 1
    return n

print(f"C fabrications (jobs affected, of {len(grounded)}):")
print(f"  Before fix: {fab_jobs_count(prefix_grounded, 'C')} of {len(prefix_grounded)}")
print(f"  After fix:  {fab_jobs_count(grounded, 'C')} of {len(grounded)}")

Variant             Before fix    After fix     
A - Minimal         3.45          2.85          
B - Simplified      2.85          3.50          
C - Full system     4.20          4.50          

C fabrications (jobs affected, of 5):
  Before fix: 0 of 5
  After fix:  0 of 5


## Findings

**1. This resume made the fabrication problem worse for A and B, not better — confirmed stable on retest.**

With a real technical background (Trixie's resume), A and B only fabricated on the 2 worst-fit jobs. With zero technical background at all, they fabricated on 3 of 5 jobs in both the original run and the retest, and invented far more per case. The bigger the real gap between candidate and job, the more a fabrication-prone system compensates by inventing qualifications.

**2. C's grounding was perfect here — 5.0 on every single job, before and after the fix.**

The gated judge found zero fabricated content across all 5 of C's outputs for this resume, in both the original run and the retest, even when that meant honestly admitting a weak fit. There was nothing here for `_enforce_fidelity` to catch — unlike the other two resumes, where it fixed (Trixie) or reduced (John Doe) real fabrication issues.

**3. B is not a safe middle ground.**

B's relative ranking flipped between runs (worst overall in the original, better than A on retest) — consistent with an unguarded prompt whose fabrication rate varies call to call. A "simplified" system without explicit anti-fabrication instructions isn't a reliably safer fallback in either direction.

**4. As a Stage 7 failure case (candidate-dependent severity), confirmed stable on retest:**

| Step | What happened |
|---|---|
| Input | Same A/B/C systems, tested on a candidate with zero relevant experience for any of the 5 jobs |
| Expected | Fabrication risk in A/B should scale with how poor the candidate-job fit is; C should stay clean regardless |
| Actual (original run) | A and B fabricated on 3/5 jobs; C stayed at zero fabrications |
| Fix (system) | Added `_enforce_fidelity` — a deterministic post-generation check stripping any skill not textually present in the source resume |
| Retest result | C remained at zero fabrications (nothing to fix on this resume); A and B continued fabricating on 3/5 jobs, confirming the original finding is a stable, candidate-dependent pattern, not a one-off |
| Implication | The anti-fabrication design matters most exactly when a candidate is least qualified — confirmed on both the original run and the retest, independent of the system fix's specific target case |